In [15]:
# this it the path to an api key for UkraineAlarm -> air_raid_api_key.txt

In [16]:
import pandas as pd

df = pd.read_parquet("data/silver_money_calc/full_dataset.parquet")
df

,EIC-код,Група,Year,Month,Day,Hour,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,АЗС,Тип,...,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m,shortwave_radiation,diffuse_radiation,direct_normal_irradiance,day_of_week,season_number,Money_spent
0,62Z0211989286227,0,2024,12,1,1,1.46219,6.136750,АЗС_23,ОККО-трасова,...,1016.9,4.2,95.0,8.6,0.0,0.0,0.0,6,1,125.359866
1,62Z0211989286227,0,2024,12,1,2,1.46219,6.136750,АЗС_23,ОККО-трасова,...,1016.6,4.5,85.0,9.0,0.0,0.0,0.0,6,1,113.472293
2,62Z0211989286227,0,2024,12,1,3,1.46219,6.136750,АЗС_23,ОККО-трасова,...,1016.4,6.0,100.0,11.9,0.0,0.0,0.0,6,1,109.149539
3,62Z0211989286227,0,2024,12,1,4,1.46219,6.136750,АЗС_23,ОККО-трасова,...,1016.0,6.4,101.0,12.6,0.0,0.0,0.0,6,1,104.826785
4,62Z0211989286227,0,2024,12,1,5,1.46219,6.136750,АЗС_23,ОККО-трасова,...,1015.7,6.4,117.0,11.9,0.0,0.0,0.0,6,1,105.907473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5454708,62Z5692449931680,0,2024,1,31,20,1.63103,3.662765,АЗС_12,ОККО-міні,...,998.5,14.7,242.0,24.5,0.0,0.0,0.0,2,1,66.806020
5454709,62Z5692449931680,0,2024,1,31,21,1.63103,3.662765,АЗС_12,ОККО-міні,...,998.4,13.8,250.0,24.1,0.0,0.0,0.0,2,1,65.278531
5454710,62Z5692449931680,0,2024,1,31,22,1.63103,3.662765,АЗС_12,ОККО-міні,...,997.9,14.3,252.0,23.8,0.0,0.0,0.0,2,1,61.243655
5454711,62Z5692449931680,0,2024,1,31,23,1.63103,3.662765,АЗС_12,ОККО-міні,...,997.3,13.9,253.0,23.8,0.0,0.0,0.0,2,1,55.767752


In [17]:
df['Область'].unique()

array(['Чернівецька', 'Закарпатська', 'Вінницька', 'Івано-Франківська',
       'Миколаївська', 'Дніпропетровська', 'Херсонська', 'Запорізька',
       'Одеська', 'Львівська', 'Черкаська', 'Чернігівська',
       'Тернопільська', 'Волинська', 'Київська', 'Хмельницька',
       'Харківська', 'м. Київ', 'Сумська', 'Кіровоградська', 'Рівненська',
       'Житомирська', 'Полтавська', 'Донецька'], dtype=object)

In [18]:
air_alarm_data_requirements = pd.DataFrame()
states = []
starts = []
ends = []

for state in df['Область'].unique().tolist():
    state_df = df[df['Область'] == state].copy()
    start = state_df['datetime'].min(axis=0)
    end = state_df['datetime'].max(axis=0)
    # print(f"State {state} has {len(state_df)} rows, starting at {start} and ending at {end}")
    states.append(state)
    starts.append(start)
    ends.append(end)

air_alarm_data_requirements['Область'] = states
air_alarm_data_requirements['Початок'] = starts
air_alarm_data_requirements['Кінець'] = ends
air_alarm_data_requirements

,Область,Початок,Кінець
0,Чернівецька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
1,Закарпатська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
2,Вінницька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
3,Івано-Франківська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
4,Миколаївська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
5,Дніпропетровська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
6,Херсонська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
7,Запорізька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
8,Одеська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00
9,Львівська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00


In [19]:
import json

with open('air_raid_regions.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# build mapping: "Чернівецька" -> "regionId"
oblast_to_id = {}

for state in data['states']:
    name = state['regionName']

    # normalize: remove " область"
    clean_name = name.replace(' область', '').strip()

    oblast_to_id[clean_name] = state['regionId']

# special case
oblast_to_id['м. Київ'] = next(
    s['regionId'] for s in data['states']
    if 'Київ' in s['regionName']
)

air_alarm_data_requirements['regionId'] = (
    air_alarm_data_requirements['Область']
    .map(oblast_to_id)
)
air_alarm_data_requirements

,Область,Початок,Кінець,regionId
0,Чернівецька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,26
1,Закарпатська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,11
2,Вінницька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,4
3,Івано-Франківська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,13
4,Миколаївська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,17
5,Дніпропетровська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,9
6,Херсонська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,23
7,Запорізька,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,12
8,Одеська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,18
9,Львівська,2024-01-01 01:00:00+02:00,2025-09-01 00:00:00+03:00,27


In [ ]:
[
  {
    "regionId": "27",
    "regionName": "Львівська область",
    "alarms": [
      {
        "regionId": "27",
        "startDate": "2025-11-02T21:36:30Z",
        "endDate": "2025-11-02T22:07:31Z",
        "duration": "00:31:01",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-30T02:28:27Z",
        "endDate": "2025-10-30T08:02:31Z",
        "duration": "05:34:04",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-27T16:35:45Z",
        "endDate": "2025-10-27T16:52:13Z",
        "duration": "00:16:28",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-25T01:03:17Z",
        "endDate": "2025-10-25T01:29:01Z",
        "duration": "00:25:44",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-22T04:03:01Z",
        "endDate": "2025-10-22T06:19:31Z",
        "duration": "02:16:30",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-20T09:19:31Z",
        "endDate": "2025-10-20T09:44:05Z",
        "duration": "00:24:34",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-16T03:19:28Z",
        "endDate": "2025-10-16T03:44:00Z",
        "duration": "00:24:32",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-16T02:19:13Z",
        "endDate": "2025-10-16T03:02:00Z",
        "duration": "00:42:47",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-14T08:26:56Z",
        "endDate": "2025-10-14T08:50:45Z",
        "duration": "00:23:49",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-10T03:23:49Z",
        "endDate": "2025-10-10T03:57:44Z",
        "duration": "00:33:55",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-10T01:52:55Z",
        "endDate": "2025-10-10T02:51:13Z",
        "duration": "00:58:18",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-05T00:50:18Z",
        "endDate": "2025-10-05T06:00:13Z",
        "duration": "05:09:55",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      },
      {
        "regionId": "27",
        "startDate": "2025-10-03T17:44:34Z",
        "endDate": "2025-10-03T18:15:54Z",
        "duration": "00:31:20",
        "alertType": "AIR",
        "regionName": "Львівська область",
        "isContinue": false
      }
    ]
  }
]

In [21]:
air_alarm_data_requirements = air_alarm_data_requirements.head(1).copy() # testing

In [22]:
air_raids_df = build_air_raid_dataset(air_alarm_data_requirements)

print("\n✅ Raw dataset:")
print(air_raids_df.head())

# save raw
save_cache(air_raids_df)

# build hourly features
hourly_df = build_hourly_features(air_raids_df)

print("\n✅ Hourly dataset:")
print(hourly_df.head())

hourly_df.to_parquet("data/air_raid_data/air_raids_hourly.parquet", index=False)

📅 Fetching from 2024-01-01 to 2025-09-01
➡️ 20240101
➡️ 20240102
➡️ 20240103
⚠️ Error 401 for 20240103
⚠️ Error 401 for 20240103
⚠️ Error 401 for 20240103
➡️ 20240104
➡️ 20240105
⚠️ Error 401 for 20240105
⚠️ Error 401 for 20240105
⚠️ Error 401 for 20240105
➡️ 20240106
⚠️ Error 401 for 20240106
⚠️ Error 401 for 20240106
⚠️ Error 401 for 20240106
➡️ 20240107
⚠️ Error 401 for 20240107
⚠️ Error 401 for 20240107
⚠️ Error 401 for 20240107
➡️ 20240108
⚠️ Error 401 for 20240108
⚠️ Error 401 for 20240108
⚠️ Error 401 for 20240108
➡️ 20240109
⚠️ Error 401 for 20240109
⚠️ Error 401 for 20240109
⚠️ Error 401 for 20240109
➡️ 20240110
⚠️ Error 401 for 20240110
⚠️ Error 401 for 20240110
⚠️ Error 401 for 20240110
➡️ 20240111
⚠️ Error 401 for 20240111
⚠️ Error 401 for 20240111
⚠️ Error 401 for 20240111
➡️ 20240112
⚠️ Error 401 for 20240112
⚠️ Error 401 for 20240112
⚠️ Error 401 for 20240112
➡️ 20240113
⚠️ Error 401 for 20240113
⚠️ Error 401 for 20240113
⚠️ Error 401 for 20240113
➡️ 20240114
⚠️ Error 40

KeyboardInterrupt: 

In [25]:
fetch_day("20240110")

[]

In [26]:
fetch_day("20250403")

[]